# 02. 대시보드 데이터 구축

-최종 04_1 취약지수와 DBSCAN 10% 권역을 기준으로 대시보드 데이터를 생성한다.
-상권시설 후보는 자동포함 자료를 기본으로 사용하고, 수동검토 자료는 문화관련성점수 2점 이상만 보조 후보로 사용한다.
-취약권역 분석은 5개 지표를 모두 제공하고, 추천 시뮬레이션은 문화분류 기반 후보 추천만 제공한다.


## 전처리 특이사항

-수동검토 1점 후보는 문화 관련성이 약해 추천 후보에서 제외한다.
-후보시설의 실제 수용능력 자료가 없으므로 가상공급량은 1로 둔다.
-후보 추가 후 접근성 전후 재계산이 필요한 값은 대시보드에서 확정값처럼 표시하지 않는다.


In [ ]:
from __future__ import annotations

import json
import math
import re
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point


try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd()
    if ROOT.name == "dashboard":
        ROOT = ROOT.parents[1]

RAW_CAND_DIR = ROOT / "data" / "raw" / "franchise_candidates" / "processed"
GRID_DIR = ROOT / "data" / "grid"
ACCESS_DIR = ROOT / "notebooks" / "access" / "OUTPUT"
DASH_DIR = ROOT / "notebooks" / "dashboard"
DASH_OUT = DASH_DIR / "OUTPUT" / "dashboard_data"
VUL_OUT = DASH_DIR / "OUTPUT" / "vulnerability_index"
DOCS_DIR = ROOT / "docs"
SHARE_DIR = DOCS_DIR / "share"
ASSET_JS = DOCS_DIR / "assets" / "dashboard_spatial_layers.js"

FINAL_GRID = ACCESS_DIR / "final_vulnerability_index" / "종합문화취약지수_선호반영_H3SFCA.csv"
SFCA_GRID = ACCESS_DIR / "final_vulnerability_index" / "종합문화취약지수_선호미반영_SFCA.csv"
DBSCAN_GRID = VUL_OUT / "DBSCAN_취약격자.csv"
DBSCAN_CLUSTER = VUL_OUT / "DBSCAN_취약권역.csv"
DASH_GRID = VUL_OUT / "격자별_최종취약지수.csv"
GRID_GPKG = ROOT / "analysis_table" / "data" / "output" / "서울시_격자_100m_행정동_기본테이블.gpkg"
BOUNDARY_GPKG = ROOT / "analysis_table" / "data" / "output" / "서울시_시군구_행정동_경계.gpkg"
H3_ACCESS = ACCESS_DIR / "h3sfca" / "h3sfca_격자_중분류_접근성.csv"
SFCA_ACCESS = ACCESS_DIR / "h3sfca" / "sfca_no_preference_격자_중분류_접근성.csv"
MERCHANT_SUPPLY = ACCESS_DIR / "h3sfca" / "문화누리_가맹점_카테고리별_공급량.csv"
TEMPLATE_HTML = DOCS_DIR / "mnc_dashboard_mockup.html"
FINAL_HTML = DOCS_DIR / "mnc_dashboard.html"
SHARE_HTML = SHARE_DIR / "mnc_dashboard_share.html"
SHARE_ZIP = SHARE_DIR / "mnc_dashboard_share.zip"
BUILD_DOC = DASH_DIR / "docs" / "dashboard_data_build_전처리_사용데이터.txt"
NOTEBOOK_PATH = DASH_DIR / "02_dashboard_data_build.ipynb"


METRIC_ORDER = ["종합취약", "시설접근성취약", "문화다양성취약", "장애인친화취약", "노인편의취약"]
METRIC_LABEL = {
    "종합취약": "종합취약도",
    "시설접근성취약": "시설접근성 취약도",
    "문화다양성취약": "문화다양성 부족도",
    "장애인친화취약": "장애인친화 취약도",
    "노인편의취약": "노인편의 취약도",
}
METRIC_PREFIX = {
    "종합취약": "종합",
    "시설접근성취약": "시설",
    "문화다양성취약": "다양성",
    "장애인친화취약": "장애인",
    "노인편의취약": "노인",
}
CATEGORY_ORDER = ["공연", "미술", "도서", "영상", "문화체험", "체육시설", "체육용품", "음악", "스포츠관람", "관광지"]
VUL_COLS = [
    "최종취약지수",
    "시설분류_접근성취약도",
    "문화다양성부족도",
    "장애인친화시설_접근성취약도",
    "노인편의서비스_접근성취약도",
]


def read_csv(path: Path, **kwargs) -> pd.DataFrame:
    return pd.read_csv(path, encoding="utf-8-sig", **kwargs)


def write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def clean_records(df: pd.DataFrame) -> list[dict]:
    return json.loads(df.where(pd.notna(df), None).to_json(orient="records", force_ascii=False))


def clean_obj(obj):
    if isinstance(obj, dict):
        return {k: clean_obj(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [clean_obj(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        if not np.isfinite(obj):
            return None
        return float(obj)
    if isinstance(obj, float) and not math.isfinite(obj):
        return None
    if pd.isna(obj) if obj is not None and not isinstance(obj, (str, list, dict)) else False:
        return None
    return obj


def to_geojson(gdf: gpd.GeoDataFrame) -> dict:
    if gdf.empty:
        return {"type": "FeatureCollection", "features": []}
    out = gdf.copy()
    for col in out.columns:
        if col != out.geometry.name and pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].astype(str)
    return json.loads(out.to_crs(4326).to_json(drop_id=True, na="null"))


def safe_sum(series: pd.Series) -> float:
    return float(pd.to_numeric(series, errors="coerce").fillna(0).sum())


def weighted_mean(value: pd.Series, weight: pd.Series) -> float:
    v = pd.to_numeric(value, errors="coerce")
    w = pd.to_numeric(weight, errors="coerce").fillna(0)
    mask = v.notna() & (w > 0)
    if not mask.any():
        return float(v.mean()) if v.notna().any() else np.nan
    return float(np.average(v[mask], weights=w[mask]))


def move_candidate_inputs() -> None:
    RAW_CAND_DIR.mkdir(parents=True, exist_ok=True)
    for name in ["중분류_자동포함.csv", "중분류_수동검토.csv"]:
        src = GRID_DIR / name
        dst = RAW_CAND_DIR / name
        if src.exists() and not dst.exists():
            shutil.move(str(src), str(dst))


def add_metric_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["취약권역_ID_원본"] = out["취약권역_ID"]
    out["취약권역_ID"] = out.apply(
        lambda r: f"{METRIC_PREFIX.get(r['권역유형'], r['권역유형'])}_{r['취약권역_ID_원본']}",
        axis=1,
    )
    out["지표라벨"] = out["권역유형"].map(METRIC_LABEL).fillna(out["권역유형"])
    return out


def derive_cluster_weak_categories(vul_grid: pd.DataFrame, final_grid: pd.DataFrame) -> pd.DataFrame:
    weak_cols = [
        "GRID_CD",
        "시설접근성_최취약중분류",
        "장애인친화시설_부족중분류",
        "노인편의서비스_부족중분류",
    ]
    merged = vul_grid.merge(final_grid[[c for c in weak_cols if c in final_grid.columns]], on="GRID_CD", how="left")
    rows = []
    for cid, part in merged.groupby("취약권역_ID", dropna=True):
        cats = []
        if "시설접근성_최취약중분류" in part.columns:
            cats.extend(part["시설접근성_최취약중분류"].dropna().astype(str).tolist())
        for text_col in ["장애인친화시설_부족중분류", "노인편의서비스_부족중분류"]:
            if text_col not in part.columns:
                continue
            for value in part[text_col].dropna().astype(str):
                cats.extend([x.strip() for x in value.split(",") if x.strip()])
        counts = pd.Series(cats, dtype="object").value_counts()
        top = counts.index.tolist()[:3]
        rows.append(
            {
                "취약권역_ID": cid,
                "부족중분류_1": top[0] if len(top) > 0 else None,
                "부족중분류_2": top[1] if len(top) > 1 else None,
                "부족중분류_3": top[2] if len(top) > 2 else None,
            }
        )
    return pd.DataFrame(rows)


def build_vulnerability_data(final_grid: pd.DataFrame, dash_grid: pd.DataFrame) -> dict:
    cluster = add_metric_keys(read_csv(DBSCAN_CLUSTER))
    vul_grid = add_metric_keys(read_csv(DBSCAN_GRID))
    grid_base = gpd.read_file(GRID_GPKG)[["GRID_CD", "geometry"]]
    if grid_base.crs is None:
        grid_base = grid_base.set_crs(5179)

    weak = derive_cluster_weak_categories(vul_grid[vul_grid["DBSCAN_label"] >= 0], final_grid)
    cluster = cluster.merge(weak, on="취약권역_ID", how="left")
    cluster["취약선정비율"] = "상위 10%"
    cluster["최우선취약권역"] = "최우선취약권역"
    cluster.loc[cluster["취약권역등급"].isna(), "취약권역등급"] = "일반취약권역"

    grid_enriched = vul_grid.merge(final_grid, on="GRID_CD", how="left", suffixes=("", "_final"))
    for col in ["시군구", "행정동", "중심점_x", "중심점_y", "문화누리대상자_추정인구수", *VUL_COLS]:
        alt = f"{col}_final"
        if alt in grid_enriched.columns:
            grid_enriched[col] = grid_enriched[col].combine_first(grid_enriched[alt])
    grid_enriched["고립취약격자여부"] = grid_enriched["DBSCAN_label"].eq(-1)
    grid_enriched["지표라벨"] = grid_enriched["권역유형"].map(METRIC_LABEL).fillna(grid_enriched["권역유형"])

    cluster_grids = grid_enriched[~grid_enriched["고립취약격자여부"]].copy()
    cluster_gdf = grid_base.merge(cluster_grids[["GRID_CD", "취약권역_ID", "권역유형"]], on="GRID_CD", how="inner")
    dissolved = cluster_gdf.dissolve(by=["권역유형", "취약권역_ID"], as_index=False)
    cluster_gdf = dissolved.merge(cluster, on=["권역유형", "취약권역_ID"], how="left")

    grid_gdf = grid_base.merge(grid_enriched, on="GRID_CD", how="inner")
    metric_cluster_geo = to_geojson(cluster_gdf)
    metric_grid_geo = to_geojson(grid_gdf)
    cluster_geo = {
        "type": "FeatureCollection",
        "features": [f for f in metric_cluster_geo["features"] if f["properties"].get("권역유형") == "종합취약"],
    }
    grid_geo = {
        "type": "FeatureCollection",
        "features": [f for f in metric_grid_geo["features"] if f["properties"].get("권역유형") == "종합취약"],
    }

    factor_stats = {}
    for _, row in cluster.iterrows():
        cid = row["취약권역_ID"]
        factor_stats[cid] = {
            "취약격자수": row.get("포함_취약격자수"),
            "권역면적_m2": row.get("권역면적_m2"),
            "시설분류_접근성취약도": row.get("평균_시설분류_접근성취약도"),
            "문화다양성부족도": row.get("평균_문화다양성부족도"),
            "장애인친화시설_접근성취약도": row.get("평균_장애인친화시설_접근성취약도"),
            "노인편의서비스_접근성취약도": row.get("평균_노인편의서비스_접근성취약도"),
        }

    grid_factor_stats = {}
    grid_cols = ["GRID_CD", *VUL_COLS]
    for _, row in dash_grid[grid_cols].dropna(subset=["GRID_CD"]).iterrows():
        grid_factor_stats[row["GRID_CD"]] = {c: row.get(c) for c in VUL_COLS}

    medians = dash_grid[dash_grid["분석대상여부"].astype(bool)][VUL_COLS].median(numeric_only=True).to_dict()

    summary_rows = []
    for metric in METRIC_ORDER:
        target = grid_enriched[grid_enriched["권역유형"] == metric]
        grouped = cluster[cluster["권역유형"] == metric]
        summary_rows.append(
            {
                "권역유형": metric,
                "지표라벨": METRIC_LABEL[metric],
                "취약권역 수": int(grouped["취약권역_ID"].nunique()),
                "DBSCAN 대상 격자": int(len(target)),
                "고립 격자": int(target["고립취약격자여부"].sum()),
                "권역 포함 격자": int((~target["고립취약격자여부"]).sum()),
            }
        )

    main_cluster = cluster[cluster["권역유형"] == "종합취약"].copy()
    main_cluster = main_cluster.sort_values(
        ["수요가중_최종취약지수", "문화누리대상자_추정인구수"],
        ascending=[False, False],
    )
    vulnerability_top5 = main_cluster.head(5)

    write_csv(grid_enriched, DASH_OUT / "dashboard_vulnerability_grid.csv")
    write_csv(cluster, DASH_OUT / "dashboard_vulnerability_cluster.csv")
    write_csv(vulnerability_top5, DASH_OUT / "dashboard_vulnerability_cluster_top5.csv")
    write_csv(pd.DataFrame(summary_rows), DASH_OUT / "dashboard_vulnerability_metric_summary.csv")

    return {
        "cluster": cluster,
        "grid_enriched": grid_enriched,
        "cluster_gdf_5179": cluster_gdf,
        "clusterGeo": cluster_geo,
        "gridGeo": grid_geo,
        "metricClusterGeo": metric_cluster_geo,
        "metricGridGeo": metric_grid_geo,
        "clusterFactorStats": clean_obj(factor_stats),
        "gridFactorStats": clean_obj(grid_factor_stats),
        "gridFactorMedians": clean_obj(medians),
        "vulnerabilityTop5": clean_records(vulnerability_top5),
        "metricSummary": summary_rows,
    }


def build_region_summaries(dash_grid: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    analysis = dash_grid[dash_grid["분석대상여부"].astype(bool)].copy()

    def summarize(part: pd.DataFrame, key_cols: list[str]) -> pd.Series:
        very = part["취약등급"].astype(str).eq("매우취약")
        vulnerable = part["취약등급"].astype(str).isin(["매우취약", "취약"])
        return pd.Series(
            {
                "분석대상격자수": len(part),
                "문화누리대상자_추정인구수": safe_sum(part["문화누리대상자_추정인구수"]),
                "취약격자수": int(vulnerable.sum()),
                "매우취약격자수": int(very.sum()),
                "취약격자비율": float(vulnerable.mean()) if len(part) else np.nan,
                "평균_최종취약지수": float(part["최종취약지수"].mean()),
                "대상자가중_최종취약지수": weighted_mean(part["최종취약지수"], part["문화누리대상자_추정인구수"]),
                "평균_시설분류_접근성취약도": float(part["시설분류_접근성취약도"].mean()),
                "평균_문화다양성부족도": float(part["문화다양성부족도"].mean()),
                "평균_장애인친화시설_접근성취약도": float(part["장애인친화시설_접근성취약도"].mean()),
                "평균_노인편의서비스_접근성취약도": float(part["노인편의서비스_접근성취약도"].mean()),
            }
        )

    gu = analysis.groupby("시군구", dropna=False).apply(lambda x: summarize(x, ["시군구"])).reset_index()
    dong = analysis.groupby(["시군구", "행정동"], dropna=False).apply(lambda x: summarize(x, ["시군구", "행정동"])).reset_index()
    write_csv(gu, DASH_OUT / "dashboard_vulnerability_gu_summary.csv")
    write_csv(dong, DASH_OUT / "dashboard_vulnerability_dong_summary.csv")

    boundary = gpd.read_file(BOUNDARY_GPKG)
    if boundary.crs is None:
        boundary = boundary.set_crs(5186)
    gu_boundary = boundary.dissolve(by="시군구", as_index=False)[["시군구", "geometry"]]
    gu_geo = to_geojson(gu_boundary.merge(gu, on="시군구", how="left"))
    return gu, dong, gu_geo


def build_category_access() -> tuple[pd.DataFrame, pd.DataFrame]:
    frames = []
    for path, model_label in [(H3_ACCESS, "선호 반영 H3SFCA"), (SFCA_ACCESS, "선호 미반영 SFCA")]:
        if not path.exists():
            continue
        df = read_csv(path)
        df["접근성모형"] = model_label
        df["접근수단"] = "분류별 기본 이동권"
        pop_col = "문화누리대상자_추정_인구수" if "문화누리대상자_추정_인구수" in df.columns else "문화누리대상자_추정인구수"
        df = df[pd.to_numeric(df[pop_col], errors="coerce").fillna(0) > 0].copy()
        frames.append(df)
    access = pd.concat(frames, ignore_index=True)
    city = (
        access.groupby(["중분류", "접근성모형", "접근수단"], dropna=False)
        .agg(
            평균_접근성지수=("접근성지수", "mean"),
            중앙값_접근성지수=("접근성지수", "median"),
            평균_접근가능가맹점수=("접근가능_가맹점수", "mean"),
            평균_선호수요=("선호수요", "mean"),
            총수요량=("수요량", "sum"),
            분석격자수=("GRID_CD", "count"),
        )
        .reset_index()
    )
    gu = (
        access.groupby(["시군구", "중분류", "접근성모형", "접근수단"], dropna=False)
        .agg(
            평균_접근성지수=("접근성지수", "mean"),
            중앙값_접근성지수=("접근성지수", "median"),
            평균_접근가능가맹점수=("접근가능_가맹점수", "mean"),
            평균_선호수요=("선호수요", "mean"),
            총수요량=("수요량", "sum"),
            분석격자수=("GRID_CD", "count"),
        )
        .reset_index()
        .rename(columns={"시군구": "시군구_격자"})
    )
    write_csv(city, DASH_OUT / "dashboard_category_access_city.csv")
    write_csv(gu, DASH_OUT / "dashboard_category_access_gu.csv")
    return city, gu


def normalize_text(s: pd.Series) -> pd.Series:
    return (
        s.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", "", regex=True)
        .str.replace(r"[^0-9a-z가-힣]", "", regex=True)
    )


def load_candidates() -> tuple[pd.DataFrame, dict]:
    auto = read_csv(RAW_CAND_DIR / "중분류_자동포함.csv")
    manual = read_csv(RAW_CAND_DIR / "중분류_수동검토.csv")
    manual["문화관련성점수"] = pd.to_numeric(manual["문화관련성점수"], errors="coerce")
    manual_use = manual[manual["문화관련성점수"] >= 2].copy()
    auto = auto.copy()
    auto["후보출처"] = "자동포함"
    auto["후보신뢰등급"] = "자동포함"
    auto["후보신뢰가중치"] = 1.0
    manual_use["후보출처"] = "수동검토"
    manual_use["후보신뢰등급"] = "수동검토_점수2"
    manual_use["후보신뢰가중치"] = 0.7
    candidates = pd.concat([auto, manual_use], ignore_index=True)
    candidates = candidates.sort_values("후보신뢰가중치", ascending=False)
    candidates = candidates.drop_duplicates("상가업소번호", keep="first")
    candidates = candidates[candidates["시도명"].eq("서울특별시")].copy()
    candidates = candidates[pd.to_numeric(candidates["경도"], errors="coerce").between(126, 128)]
    candidates = candidates[pd.to_numeric(candidates["위도"], errors="coerce").between(37, 38)]
    candidates["후보중분류"] = candidates["문화중분류"]
    candidates["가상공급량"] = 1.0

    if MERCHANT_SUPPLY.exists():
        existing = read_csv(MERCHANT_SUPPLY)
        existing_key = normalize_text(existing["가맹점명"]) + "|" + normalize_text(existing["주소"])
        cand_key = normalize_text(candidates["상호명"]) + "|" + normalize_text(candidates["도로명주소"])
        candidates = candidates[~cand_key.isin(set(existing_key))].copy()

    summary = {
        "자동포함_원행수": int(len(auto)),
        "수동검토_원행수": int(len(manual)),
        "수동검토_활용행수": int(len(manual_use)),
        "수동검토_제외행수": int(len(manual) - len(manual_use)),
        "최종후보수": int(len(candidates)),
    }
    write_csv(candidates, DASH_OUT / "dashboard_candidate_facility.csv")
    return candidates, summary


def build_cluster_category_demand(vul_grid: pd.DataFrame) -> pd.DataFrame:
    if not H3_ACCESS.exists():
        return pd.DataFrame(columns=["취약권역_ID", "중분류", "권역_선호수요", "권역_수요량", "권역_평균접근성"])
    h3 = read_csv(H3_ACCESS, usecols=["GRID_CD", "중분류", "접근성지수", "선호수요", "수요량"])
    base = vul_grid[(vul_grid["권역유형"] == "종합취약") & (vul_grid["DBSCAN_label"] >= 0)][["GRID_CD", "취약권역_ID"]]
    merged = h3.merge(base, on="GRID_CD", how="inner")
    return (
        merged.groupby(["취약권역_ID", "중분류"], dropna=False)
        .agg(
            권역_선호수요=("선호수요", "sum"),
            권역_수요량=("수요량", "sum"),
            권역_평균접근성=("접근성지수", "mean"),
        )
        .reset_index()
    )


def build_candidate_recommendations(candidates: pd.DataFrame, vulnerability: dict) -> tuple[pd.DataFrame, dict]:
    cluster = vulnerability["cluster"]
    grid_enriched = vulnerability["grid_enriched"]
    cluster_gdf = vulnerability["cluster_gdf_5179"]
    main_cluster = cluster_gdf[cluster_gdf["권역유형"] == "종합취약"].copy()
    main_cluster = main_cluster[
        [
            "취약권역_ID",
            "주요_시군구",
            "주요_행정동",
            "문화누리대상자_추정인구수",
            "평균_우선지원지수",
            "수요가중_최종취약지수",
            "부족중분류_1",
            "부족중분류_2",
            "부족중분류_3",
            "geometry",
        ]
    ].copy()

    points = gpd.GeoDataFrame(
        candidates.copy(),
        geometry=[
            Point(float(x), float(y))
            for x, y in zip(candidates["경도"], candidates["위도"])
        ],
        crs=4326,
    ).to_crs(5179)

    joined = gpd.sjoin_nearest(
        points,
        main_cluster,
        how="inner",
        max_distance=1500,
        distance_col="취약권역까지거리_m",
    ).drop(columns=["index_right"])

    demand = build_cluster_category_demand(grid_enriched)
    joined = joined.merge(
        demand,
        left_on=["취약권역_ID", "후보중분류"],
        right_on=["취약권역_ID", "중분류"],
        how="left",
    )

    def match_score(row: pd.Series) -> float:
        cat = row.get("후보중분류")
        if cat == row.get("부족중분류_1"):
            return 1.0
        if cat == row.get("부족중분류_2"):
            return 0.8
        if cat == row.get("부족중분류_3"):
            return 0.6
        return 0.35

    joined["부족중분류일치점수"] = joined.apply(match_score, axis=1)
    joined["권역_선호수요"] = pd.to_numeric(joined["권역_선호수요"], errors="coerce").fillna(0)
    joined["권역_수요량"] = pd.to_numeric(joined["권역_수요량"], errors="coerce").fillna(0)
    joined["취약권역까지거리_m"] = pd.to_numeric(joined["취약권역까지거리_m"], errors="coerce").fillna(1500)
    joined["거리감쇠점수"] = 1 / (1 + joined["취약권역까지거리_m"] / 700)
    joined["기대수요점수"] = np.log1p(joined["권역_선호수요"].clip(lower=0) + joined["권역_수요량"].clip(lower=0) * 0.1)
    joined["추천우선점수"] = (
        pd.to_numeric(joined["수요가중_최종취약지수"], errors="coerce").fillna(0)
        * (1 + joined["기대수요점수"])
        * joined["부족중분류일치점수"]
        * pd.to_numeric(joined["후보신뢰가중치"], errors="coerce").fillna(0.7)
        * joined["거리감쇠점수"]
    )
    joined["추천근거"] = joined.apply(
        lambda r: f"{r['주요_시군구']} {r['주요_행정동']} 취약권역의 {r['후보중분류']} 후보. 부족분류 일치 {r['부족중분류일치점수']:.1f}, 거리 {r['취약권역까지거리_m']:.0f}m.",
        axis=1,
    )
    joined["접근성_모형"] = "선호 반영 H3SFCA"
    joined["후보평가상태"] = "프로토타입 우선순위"
    joined = joined.sort_values("추천우선점수", ascending=False)
    top = joined.head(300).copy()
    keep_cols = [
        "상가업소번호",
        "상호명",
        "후보중분류",
        "문화세부분류",
        "상권업종중분류명",
        "상권업종소분류명",
        "시군구명",
        "행정동명",
        "도로명주소",
        "경도",
        "위도",
        "후보출처",
        "후보신뢰등급",
        "후보신뢰가중치",
        "가상공급량",
        "취약권역_ID",
        "주요_시군구",
        "주요_행정동",
        "문화누리대상자_추정인구수",
        "평균_우선지원지수",
        "수요가중_최종취약지수",
        "부족중분류_1",
        "부족중분류_2",
        "부족중분류_3",
        "부족중분류일치점수",
        "권역_선호수요",
        "권역_수요량",
        "권역_평균접근성",
        "취약권역까지거리_m",
        "거리감쇠점수",
        "추천우선점수",
        "추천근거",
        "접근성_모형",
        "후보평가상태",
    ]
    top_out = top[[c for c in keep_cols if c in top.columns]].copy()
    write_csv(top_out, DASH_OUT / "dashboard_candidate_recommendation_top300.csv")

    cat_summary = (
        candidates.groupby(["후보중분류", "후보신뢰등급"], dropna=False)
        .size()
        .reset_index(name="후보수")
        .sort_values(["후보중분류", "후보신뢰등급"])
    )
    write_csv(cat_summary, DASH_OUT / "dashboard_candidate_category_summary.csv")
    top_geo = to_geojson(top.set_geometry("geometry"))
    return top_out, top_geo


def parse_existing_dashboard_data() -> dict:
    text = TEMPLATE_HTML.read_text(encoding="utf-8")
    match = re.search(r'<script type="application/json" id="dashboardData">(.*?)</script>', text, re.S)
    if not match:
        raise RuntimeError("dashboardData script not found")
    return json.loads(match.group(1))


def patch_dashboard_html(data: dict) -> None:
    html = TEMPLATE_HTML.read_text(encoding="utf-8")
    data_json = json.dumps(clean_obj(data), ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
    html = re.sub(
        r'(<script type="application/json" id="dashboardData">).*?(</script>)',
        r"\1" + data_json + r"\2",
        html,
        flags=re.S,
    )

    html = html.replace(
        '<label class="vul-filter-only">지역 검색</label>\n                    <input class="map-search-input" id="regionSearchInput"',
        '<label class="vul-filter-only">분석지표</label>\n'
        '                    <select id="vulMetricSelect" class="map-select vul-filter-only"></select>\n'
        '                    <label class="vul-filter-only">지역 검색</label>\n'
        '                    <input class="map-search-input" id="regionSearchInput"',
    )

    html = html.replace(
        'let selectedCategory = "\\uc804\\uccb4";\n    let selectedGuName',
        'let selectedCategory = "\\uc804\\uccb4";\n    let selectedVulMetric = "종합취약";\n    let selectedGuName',
    )

    helper = r'''
    function activeMetricMeta() {
      const rows = DATA.vulnerabilityMetrics || [];
      return rows.find(r => r.key === selectedVulMetric) || rows[0] || {key:"종합취약", label:"종합취약도"};
    }
    function activeMetricLabel() {
      return activeMetricMeta().label || selectedVulMetric;
    }
    function activeClusterFeatures() {
      const source = DATA.metricClusterGeo || DATA.clusterGeo || {features:[]};
      return (source.features || []).filter(f => !selectedVulMetric || f.properties?.권역유형 === selectedVulMetric);
    }
    function activeGridFeatures() {
      const source = DATA.metricGridGeo || DATA.gridGeo || {features:[]};
      return (source.features || []).filter(f => !selectedVulMetric || f.properties?.권역유형 === selectedVulMetric);
    }
    function activeClusterGeo() {
      return {type:"FeatureCollection", features:activeClusterFeatures()};
    }
    function activeGridGeo() {
      return {type:"FeatureCollection", features:activeGridFeatures()};
    }
    function renderMetricControls() {
      const rows = DATA.vulnerabilityMetrics || [];
      const options = rows.map(r => `<option value="${safe(r.key)}">${safe(r.label)}</option>`).join("");
      if ($("#vulMetricSelect")) {
        $("#vulMetricSelect").innerHTML = options;
        $("#vulMetricSelect").value = selectedVulMetric;
      }
    }
    function makeClusterLayer() {
      return L.geoJSON(activeClusterGeo(), {
        pane:"clusterPane",
        style: clusterStyle,
        onEachFeature: (f,l) => {
          const p = f.properties;
          l.bindTooltip(`<b>${safe(p[K.clusterId])} · ${safe(p[K.mainGu])} ${safe(p[K.mainDong])}</b><br>${safe(p.지표라벨 || activeMetricLabel())}<br>대상자 ${fmt.n(p[K.mncPop])}명<br>취약지수 ${fmt.two(p[K.demandScore])}<br>부족분류 ${[p[K.lack1],p[K.lack2],p[K.lack3]].filter(Boolean).map(safe).join(" · ")}`, {sticky:true});
          l.on("mouseover", () => l.setStyle({weight:3.5, fillOpacity:.3, color: selectedCategory === "\uc804\uccb4" ? "#ff5b1f" : categoryColor(selectedCategory)}));
          l.on("mouseout", () => l.setStyle(clusterStyle(f)));
          l.on("click", e => { L.DomEvent.stopPropagation(e); runWithLoading(() => selectCluster(p), {messages: selectionLoadingMessages}); });
        }
      });
    }
    function makeGridLayer() {
      return L.geoJSON(activeGridGeo(), {
        pane:"gridPane",
        style: gridStyle,
        onEachFeature: (f,l) => {
          const p = f.properties;
          const isolated = p.고립취약격자여부 ? " · 고립 취약격자" : "";
          l.bindTooltip(`<b>${safe(p.GRID_CD)}</b><br>${safe(p[K.gu])} ${safe(p[K.dong])}<br>${safe(p.지표라벨 || activeMetricLabel())}${isolated}<br>${safe(p[K.gridGrade])} · 대상자 ${fmt.n(p[K.mncPop])}명<br>취약지수 ${fmt.two(p[K.gridScore])}`, {sticky:true});
          l.on("mouseover", () => l.setStyle({weight:1.05, color:"#ff6a1f", fillOpacity:.46}));
          l.on("mouseout", () => l.setStyle(gridStyle(f)));
          l.on("click", e => { L.DomEvent.stopPropagation(e); runWithLoading(() => selectGridFeature(p), {messages: selectionLoadingMessages}); });
        }
      });
    }
    function refreshVulnerabilityMetric() {
      const keepCluster = clusterLayer && map.hasLayer(clusterLayer);
      const keepGrid = gridLayer && map.hasLayer(gridLayer);
      if (clusterLayer) map.removeLayer(clusterLayer);
      if (gridLayer) map.removeLayer(gridLayer);
      selectedClusterProps = null;
      clusterLayer = makeClusterLayer();
      gridLayer = makeGridLayer();
      if (keepCluster || $("#clusterToggle")?.classList.contains("active")) clusterLayer.addTo(map);
      if (keepGrid && $("#gridToggle")?.classList.contains("active")) gridLayer.addTo(map);
      renderVulnerability();
      updateMapLegend();
      applyLayers();
      const first = activeClusterFeatures()[0];
      if (first?.properties) selectCluster(first.properties);
    }
'''
    html = html.replace("    function categoryColor(cat) {", helper + "\n    function categoryColor(cat) {")

    old_cluster = re.search(
        r"      clusterLayer = L\.geoJSON\(DATA\.clusterGeo, \{.*?      \}\)\.addTo\(map\);\n      gridLayer = L\.geoJSON\(DATA\.gridGeo, \{.*?      \}\);\n",
        html,
        re.S,
    )
    if old_cluster:
        html = html[: old_cluster.start()] + "      clusterLayer = makeClusterLayer().addTo(map);\n      gridLayer = makeGridLayer();\n" + html[old_cluster.end() :]

    html = html.replace(
        "      selectCluster(DATA.clusterGeo.features.find(f => f.properties[K.clusterId] === DATA.vulnerabilityTop5[0]?.[K.clusterId])?.properties || DATA.clusterGeo.features[0].properties);",
        "      renderMetricControls();\n      selectCluster(activeClusterFeatures().find(f => f.properties[K.clusterId] === DATA.vulnerabilityTop5[0]?.[K.clusterId])?.properties || activeClusterFeatures()[0]?.properties);",
    )
    html = html.replace("(DATA.clusterGeo.features || [])", "activeClusterFeatures()")
    html = html.replace("(DATA.gridGeo.features || [])", "activeGridFeatures()")
    html = html.replace("selectCluster(DATA.clusterGeo.features[0].properties);", "selectCluster(activeClusterFeatures()[0]?.properties);")
    html = html.replace("DATA.clusterGeo.features[0]", "activeClusterFeatures()[0]")
    html = html.replace("DATA.clusterGeo.features.find(f => categoryMatchesProperties(f.properties))", "activeClusterFeatures().find(f => categoryMatchesProperties(f.properties))")
    html = html.replace("findFeatureAtLatLng(DATA.gridGeo, e.latlng)", "findFeatureAtLatLng(activeGridGeo(), e.latlng)")
    html = html.replace("findFeatureAtLatLng(DATA.clusterGeo, e.latlng)", "findFeatureAtLatLng(activeClusterGeo(), e.latlng)")
    html = html.replace(
        'const clusters = activeClusterFeatures().map(f => f.properties || {});',
        'const clusters = activeClusterFeatures().map(f => f.properties || {});',
    )
    html = html.replace("fmt.n(s.clusterCount)", "fmt.n(clusters.length)")
    html = html.replace("${fmt.n(s.clusterCount)}개 취약권역", "${fmt.n(clusters.length)}개 취약권역")
    html = html.replace("종합취약도</span>", "${safe(activeMetricLabel())}</span>")
    html = html.replace(
        'setText("#vulSub", isRec ? "선택 취약권역의 부족 분야에 맞춰 전환 후보와 기존 동일 분야 가맹점의 중복을 검토합니다." : "종합취약도가 높은 인접 격자를 권역화하고, 클릭 시 취약 원인과 우선 보강 분야를 확인합니다.");',
        'setText("#vulSub", isRec ? "선택 취약권역과 문화분류에 맞춰 상권시설 후보의 전환 우선순위를 검토합니다." : `${activeMetricLabel()} 상위 10% 격자를 DBSCAN으로 권역화하고, 클릭 시 취약 원인과 보강 분야를 확인합니다.`);',
    )
    html = html.replace(
        'setText("#mapSub", isRec ? "선택 취약권역 · 기존 동일 분야 가맹점 · 추천 후보 TOP" : "구 경계 · DBSCAN 취약권역 · 100m 취약격자");',
        'setText("#mapSub", isRec ? "선택 취약권역 · 기존 동일 분야 가맹점 · 추천 후보 TOP" : `${activeMetricLabel()} · DBSCAN 취약권역 · 100m 취약격자`);',
    )
    html = html.replace(
        'if ($("#categorySelect")) $("#categorySelect").onchange = e => runWithLoading(() => updateCategorySelection(e.target.value), {messages: selectionLoadingMessages});',
        'if ($("#categorySelect")) $("#categorySelect").onchange = e => runWithLoading(() => updateCategorySelection(e.target.value), {messages: selectionLoadingMessages});\n'
        '      if ($("#vulMetricSelect")) $("#vulMetricSelect").onchange = e => runWithLoading(() => { selectedVulMetric = e.target.value; refreshVulnerabilityMetric(); }, {messages: selectionLoadingMessages});',
    )

    FINAL_HTML.write_text(html, encoding="utf-8")

    share = html
    if ASSET_JS.exists():
        asset = ASSET_JS.read_text(encoding="utf-8")
        share = share.replace('<script src="assets/dashboard_spatial_layers.js"></script>', f"<script>\n{asset}\n</script>")
    SHARE_DIR.mkdir(parents=True, exist_ok=True)
    SHARE_HTML.write_text(share, encoding="utf-8")
    with zipfile.ZipFile(SHARE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(SHARE_HTML, arcname=SHARE_HTML.name)


def cleanup_obsolete_outputs() -> list[str]:
    removed = []
    for path in [
        DASH_OUT / "dashboard_candidate_recommendation_top300.gpkg",
        DASH_OUT / "dashboard_vulnerability_cluster.gpkg",
        DASH_OUT / "dashboard_vulnerability_grid.gpkg",
        DASH_OUT / "dashboard_cluster_category_vulnerability.csv",
        DASH_OUT / "dashboard_cluster_weak_category.csv",
        DASH_OUT / "dashboard_vulnerability_cause_summary.csv",
        SHARE_DIR / "mnc_dashboard_mockup_share.html",
        SHARE_DIR / "mnc_dashboard_mockup_share.zip",
    ]:
        if path.exists():
            path.unlink()
            removed.append(str(path.relative_to(ROOT)))
    return removed


def write_docs(candidate_summary: dict, metric_summary: list[dict], removed: list[str]) -> None:
    lines = [
        "# 대시보드 데이터 구축 메모",
        "",
        "## 사용 데이터",
        "-최종 취약지수: 종합문화취약지수_선호반영_H3SFCA.csv",
        "-취약권역: DBSCAN_취약권역.csv / DBSCAN_취약격자.csv",
        "-공간경계: 서울시 100m 격자, 시군구·행정동 경계",
        "-기존 가맹점: 문화누리 가맹점 공급량 산출 결과",
        "-상권시설 후보: 중분류_자동포함.csv, 중분류_수동검토.csv",
        "",
        "## 후보시설 전처리",
        f"-자동포함 {candidate_summary['자동포함_원행수']:,}건은 바로 후보로 사용",
        f"-수동검토 {candidate_summary['수동검토_원행수']:,}건 중 문화관련성점수 2점 이상 {candidate_summary['수동검토_활용행수']:,}건만 보조 후보로 사용",
        f"-수동검토 1점 {candidate_summary['수동검토_제외행수']:,}건은 키워드 관련성이 약해 추천 후보에서 제외",
        "-기존 문화누리 가맹점과 상호명·주소가 같은 후보는 중복 후보에서 제외",
        "-후보 공급량 자료가 없으므로 후보 1개를 표준 가상공급량 1로 처리",
        f"-최종 후보시설 {candidate_summary['최종후보수']:,}건 생성",
        "",
        "## 취약권역 반영",
    ]
    for row in metric_summary:
        lines.append(
            f"-{row['권역유형']}: 취약권역 {row['취약권역 수']:,}개, 대상격자 {row['DBSCAN 대상 격자']:,}개, 고립격자 {row['고립 격자']:,}개"
        )
    lines += [
        "",
        "## 대시보드 작동 방식",
        "-취약권역 분석: 종합·시설접근성·문화다양성·장애인친화·노인편의 지표를 선택해 별도 DBSCAN 권역 표시",
        "-가맹점 유치 시뮬레이션: 문화분류 기반 상권시설 후보만 추천",
        "-장애인·노인 지표는 취약현황 분석에는 표시하되, 후보 추천효과 계산에서는 제외",
        "-후보 추천은 취약권역 점수, 부족중분류 일치, H3SFCA 선호수요, 거리감쇠, 후보 신뢰등급을 반영한 프로토타입 우선순위",
        "-후보 추가 후 H3SFCA 전후 재계산이 필요한 신규 수혜자·접근성 개선량은 확정값으로 표시하지 않음",
        "",
        "## 주요 산출물",
        "-docs/mnc_dashboard.html",
        "-docs/share/mnc_dashboard_share.html",
        "-docs/share/mnc_dashboard_share.zip",
        "-notebooks/dashboard/OUTPUT/dashboard_data/dashboard_candidate_facility.csv",
        "-notebooks/dashboard/OUTPUT/dashboard_data/dashboard_candidate_recommendation_top300.csv",
        "-notebooks/dashboard/OUTPUT/dashboard_data/dashboard_vulnerability_cluster.csv",
        "-notebooks/dashboard/OUTPUT/dashboard_data/dashboard_vulnerability_grid.csv",
    ]
    if removed:
        lines += ["", "## 삭제한 이전 산출물", *[f"-{x}" for x in removed]]
    BUILD_DOC.parent.mkdir(parents=True, exist_ok=True)
    BUILD_DOC.write_text("\n".join(lines), encoding="utf-8")


def write_notebook(script_text: str) -> None:
    cells = [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# 02. 대시보드 데이터 구축\n",
                "\n",
                "-최종 04_1 취약지수와 DBSCAN 10% 권역을 기준으로 대시보드 데이터를 생성한다.\n",
                "-상권시설 후보는 자동포함 자료를 기본으로 사용하고, 수동검토 자료는 문화관련성점수 2점 이상만 보조 후보로 사용한다.\n",
                "-취약권역 분석은 5개 지표를 모두 제공하고, 추천 시뮬레이션은 문화분류 기반 후보 추천만 제공한다.\n",
            ],
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "## 전처리 특이사항\n",
                "\n",
                "-수동검토 1점 후보는 문화 관련성이 약해 추천 후보에서 제외한다.\n",
                "-후보시설의 실제 수용능력 자료가 없으므로 가상공급량은 1로 둔다.\n",
                "-후보 추가 후 접근성 전후 재계산이 필요한 값은 대시보드에서 확정값처럼 표시하지 않는다.\n",
            ],
        },
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": script_text.splitlines(keepends=True),
        },
    ]
    nb = {
        "cells": cells,
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3"},
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }
    NOTEBOOK_PATH.write_text(json.dumps(nb, ensure_ascii=False, indent=1), encoding="utf-8")


def validate_outputs(data: dict) -> dict:
    if not FINAL_HTML.exists() or not SHARE_HTML.exists():
        raise RuntimeError("HTML output missing")
    for key in ["clusterGeo", "metricClusterGeo", "candidateGeo"]:
        if not data.get(key, {}).get("features"):
            raise RuntimeError(f"{key} has no features")
    return {
        "html_mb": round(FINAL_HTML.stat().st_size / 1024 / 1024, 2),
        "share_html_mb": round(SHARE_HTML.stat().st_size / 1024 / 1024, 2),
        "share_zip_mb": round(SHARE_ZIP.stat().st_size / 1024 / 1024, 2),
        "cluster_features": len(data["clusterGeo"]["features"]),
        "metric_cluster_features": len(data["metricClusterGeo"]["features"]),
        "candidate_features": len(data["candidateGeo"]["features"]),
    }


def main() -> None:
    move_candidate_inputs()
    DASH_OUT.mkdir(parents=True, exist_ok=True)
    SHARE_DIR.mkdir(parents=True, exist_ok=True)

    final_grid = read_csv(FINAL_GRID)
    dash_grid = read_csv(DASH_GRID)
    old = parse_existing_dashboard_data()

    vulnerability = build_vulnerability_data(final_grid, dash_grid)
    gu, dong, gu_geo = build_region_summaries(dash_grid)
    category_city, category_gu = build_category_access()
    candidates, candidate_summary = load_candidates()
    candidate_top, candidate_geo = build_candidate_recommendations(candidates, vulnerability)

    summary = dict(old.get("summary", {}))
    top_gu = gu.sort_values("대상자가중_최종취약지수", ascending=False).iloc[0]["시군구"]
    summary.update(
        {
            "clusterCount": len(vulnerability["clusterGeo"]["features"]),
            "vulnerableGridCount": len(vulnerability["gridGeo"]["features"]),
            "candidateCount": candidate_summary["최종후보수"],
            "topGu": top_gu,
            "dashboardVersion": "04_1_H3SFCA_DBSCAN10",
            "builtAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }
    )

    data = dict(old)
    data.update(
        {
            "summary": summary,
            "vulnerabilityGu": clean_records(gu),
            "vulnerabilityDong": clean_records(dong),
            "vulnerabilityCluster": clean_records(vulnerability["cluster"][vulnerability["cluster"]["권역유형"] == "종합취약"]),
            "vulnerabilityTop5": vulnerability["vulnerabilityTop5"],
            "categoryAccessCity": clean_records(category_city),
            "categoryAccessGu": clean_records(category_gu),
            "candidateSummary": clean_records(
                candidates.groupby("후보중분류", dropna=False)
                .size()
                .reset_index(name="후보수")
                .sort_values("후보수", ascending=False)
            ),
            "candidateTop": clean_records(candidate_top),
            "guGeo": gu_geo,
            "clusterGeo": vulnerability["clusterGeo"],
            "gridGeo": vulnerability["gridGeo"],
            "metricClusterGeo": vulnerability["metricClusterGeo"],
            "metricGridGeo": vulnerability["metricGridGeo"],
            "candidateGeo": candidate_geo,
            "gridFactorMedians": vulnerability["gridFactorMedians"],
            "clusterFactorStats": vulnerability["clusterFactorStats"],
            "gridFactorStats": vulnerability["gridFactorStats"],
            "vulnerabilityMetrics": [
                {"key": key, "label": METRIC_LABEL[key], "description": f"{METRIC_LABEL[key]} 상위 10% DBSCAN 권역"}
                for key in METRIC_ORDER
            ],
            "metricSummary": vulnerability["metricSummary"],
            "candidatePolicy": candidate_summary,
        }
    )

    patch_dashboard_html(data)
    removed = cleanup_obsolete_outputs()
    write_docs(candidate_summary, vulnerability["metricSummary"], removed)
    if "__file__" in globals():
        write_notebook(Path(__file__).read_text(encoding="utf-8"))
    result = validate_outputs(data)
    print(json.dumps({"candidate_summary": candidate_summary, "removed": removed, "result": result}, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 추천 후보풀 보정

-상위 300개 전체 방식은 체육시설·H3SFCA에 과도하게 쏠려 사용하지 않음
-H3SFCA/SFCA 모형별·문화분류별·권역별로 균형 추출한 후보풀을 최종 HTML에 반영
-후보시설은 지도에 전부 올리지 않고 추천 실행 후 K개만 표시


In [ ]:
## 추천 후보풀 보정
import base64
import mimetypes
import re
import zipfile

candidate_pool_path = DASH_OUT / "dashboard_candidate_recommendation_pool.csv"
if not candidate_pool_path.exists():
    raise FileNotFoundError(f"추천 후보풀 파일이 없습니다: {candidate_pool_path}")

candidate_pool = read_csv(candidate_pool_path)


def inline_loader_images_for_share(html: str) -> str:
    def repl(match: re.Match) -> str:
        raw = match.group(1)
        path = (DOCS_DIR / raw).resolve()
        try:
            path.relative_to(DOCS_DIR.resolve())
        except ValueError:
            return match.group(0)
        if not path.exists():
            return match.group(0)
        mime = mimetypes.guess_type(path.name)[0] or "image/png"
        data = base64.b64encode(path.read_bytes()).decode("ascii")
        return f'url("data:{mime};base64,{data}")'

    return re.sub(r'url\("((?:assets/)[^"]+\.(?:png|jpg|jpeg|webp|gif))"\)', repl, html)


def rebuild_dashboard_share_from_final(final_html: str) -> None:
    share = final_html
    if ASSET_JS.exists():
        share = share.replace(
            '<script src="assets/dashboard_spatial_layers.js"></script>',
            f"<script>\n{ASSET_JS.read_text(encoding='utf-8')}\n</script>",
        )
    share = inline_loader_images_for_share(share)
    SHARE_HTML.write_text(share, encoding="utf-8")
    with zipfile.ZipFile(SHARE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(SHARE_HTML, arcname=SHARE_HTML.name)


html = FINAL_HTML.read_text(encoding="utf-8")
tag = '<script type="application/json" id="dashboardData">'
start = html.index(tag) + len(tag)
end = html.index("</script>", start)
dashboard_data = json.loads(html[start:end])
dashboard_data["candidateTop"] = clean_records(candidate_pool)
dashboard_data["candidateGeo"] = {"type": "FeatureCollection", "features": []}
dashboard_data["candidatePolicy"] = {
    **dashboard_data.get("candidatePolicy", {}),
    "추천후보풀_행수": int(len(candidate_pool)),
    "추천후보풀_모형수": int(candidate_pool["접근성_모형"].nunique()),
    "추천후보풀_분류수": int(candidate_pool["후보중분류"].nunique()),
    "추천후보풀_설계": "상위 300 전체가 아니라 H3/SFCA 모형별·문화분류별·권역별 후보를 균형 추출",
}
data_json = json.dumps(clean_obj(dashboard_data), ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
html = html[:start] + data_json + html[end:]
FINAL_HTML.write_text(html, encoding="utf-8")
rebuild_dashboard_share_from_final(html)

display(pd.DataFrame([
    {
        "추천후보풀_행수": len(candidate_pool),
        "접근성_모형수": candidate_pool["접근성_모형"].nunique(),
        "후보중분류수": candidate_pool["후보중분류"].nunique(),
        "공유HTML": str(SHARE_HTML),
    }
]))